# 03 - Validaciones Iniciales y Detección de Anomalías (Capa Bronze)

Trabajo Práctico 1 - Sección 4

Este notebook hace **diagnóstico exploratorio de solo lectura** sobre las tablas Bronze creadas en `01_Ingesta_Bronze_RUES.ipynb` y `02_Ingesta_Bronze_TRM.ipynb`. No se modifica ni se limpia ningún dato — los hallazgos aquí documentados son insumo para el diseño de la futura **Capa Silver**.

Se valida:

1. Conteo total de registros y volumen de nulos/vacíos por columna.
2. Duplicados exactos.
3. Datos atípicos / outliers específicos del dominio (fechas centinela, fechas futuras, valores fuera de rango).

In [ ]:
%python
from pyspark.sql import functions as F

df_rues = spark.table("Datos_Empresas.bronze.rues_registro_mercantil")
df_trm = spark.table("Datos_Empresas.bronze.trm_tasa_cambio")

print(f"RUES: {df_rues.count():,} filas, {len(df_rues.columns)} columnas")
print(f"TRM:  {df_trm.count():,} filas, {len(df_trm.columns)} columnas")

---

## 1. Valores nulos / vacíos por columna

En RUES, todas las columnas son texto: se cuenta tanto `NULL` como cadena vacía `""` como "sin dato".

In [ ]:
%python
def contar_nulos_y_vacios(df, columnas_texto=()):
    exprs = []
    for c in df.columns:
        if c in columnas_texto:
            cond = F.col(c).isNull() | (F.trim(F.col(c)) == "")
        else:
            cond = F.col(c).isNull()
        exprs.append(F.count(F.when(cond, c)).alias(c))
    return df.select(exprs)


print("Nulos/vacíos por columna - RUES:")
columnas_texto_rues = [c for c, t in df_rues.dtypes if t == "string"]
display(contar_nulos_y_vacios(df_rues, columnas_texto_rues))

In [ ]:
%python
print("Nulos/vacíos por columna - TRM:")
columnas_texto_trm = [c for c, t in df_trm.dtypes if t == "string"]
display(contar_nulos_y_vacios(df_trm, columnas_texto_trm))

---

## 2. Duplicados exactos

Se comparan las columnas originales de la fuente (sin contar `_ingested_at`, que siempre difiere entre corridas).

In [ ]:
%python
def contar_duplicados_exactos(df, columnas_negocio):
    total = df.count()
    unicos = df.select(columnas_negocio).dropDuplicates().count()
    return total, unicos, total - unicos


columnas_negocio_rues = [c for c in df_rues.columns if c not in ("_ingested_at", "_source")]
total, unicos, duplicados = contar_duplicados_exactos(df_rues, columnas_negocio_rues)
print(f"RUES -> total: {total:,} | únicos: {unicos:,} | duplicados exactos: {duplicados:,}")

columnas_negocio_trm = [c for c in df_trm.columns if c not in ("_ingested_at", "_source")]
total_t, unicos_t, duplicados_t = contar_duplicados_exactos(df_trm, columnas_negocio_trm)
print(f"TRM  -> total: {total_t:,} | únicos: {unicos_t:,} | duplicados exactos: {duplicados_t:,}")

In [ ]:
%python
# Adicional: duplicados por llave de negocio (una empresa no debería repetirse
# con la misma matrícula + cámara de comercio)
duplicados_por_matricula = (
    df_rues.groupBy("codigo_camara", "matricula")
    .count()
    .filter(F.col("count") > 1)
)
print(f"Combinaciones (codigo_camara, matricula) repetidas: {duplicados_por_matricula.count():,}")
display(duplicados_por_matricula.orderBy(F.desc("count")).limit(10))

---

## 3. Datos atípicos / outliers

### 3.1 RUES: fechas centinela, formatos inválidos y fechas futuras

Las fechas de RUES llegan como texto `YYYYMMDD`. Se identifican:
* Valores que no calzan con el patrón de 8 dígitos.
* El valor centinela `99991231` usado por la fuente para "sin fecha de vencimiento".
* Fechas de matrícula posteriores a hoy (inconsistentes con la realidad del negocio).

In [ ]:
%python
hoy = F.date_format(F.current_date(), "yyyyMMdd")

formato_invalido = df_rues.filter(~F.col("fecha_matricula").rlike(r"^\d{8}$"))
print(f"fecha_matricula con formato distinto a 8 dígitos: {formato_invalido.count():,}")

centinela_vigencia = df_rues.filter(F.col("fecha_vigencia") == "99991231")
print(f"fecha_vigencia con valor centinela '99991231' (sin vencimiento): {centinela_vigencia.count():,}")

matricula_futura = df_rues.filter(
    F.col("fecha_matricula").rlike(r"^\d{8}$") & (F.col("fecha_matricula") > hoy)
)
print(f"fecha_matricula en el futuro respecto a hoy: {matricula_futura.count():,}")
display(matricula_futura.select("codigo_camara", "matricula", "fecha_matricula").limit(10))

### 3.2 TRM: valores fuera de rango y rangos de vigencia inválidos

* `valor` (tasa de cambio) no debería ser cero ni negativo.
* `vigenciadesde` no debería ser posterior a `vigenciahasta`.

In [ ]:
%python
valor_invalido = df_trm.filter(F.col("valor").cast("double") <= 0)
print(f"TRM con valor <= 0: {valor_invalido.count():,}")

rango_invalido = df_trm.filter(F.col("vigenciadesde") > F.col("vigenciahasta"))
print(f"TRM con vigenciadesde > vigenciahasta: {rango_invalido.count():,}")

display(df_trm.orderBy(F.desc("vigenciadesde")).limit(5))

---

## Resumen de hallazgos (para la Capa Silver)

_Completar tras ejecutar las celdas anteriores con los números reales obtenidos:_

| Hallazgo | Fuente | Cantidad | Acción sugerida en Silver |
|---|---|---|---|
| Nulos/vacíos por columna | RUES / TRM | _ver salida_ | Definir reglas de completitud mínima por columna crítica |
| Duplicados exactos | RUES / TRM | _ver salida_ | `dropDuplicates()` en Silver, nunca en Bronze |
| `fecha_vigencia = 99991231` | RUES | _ver salida_ | Convertir a `NULL` explícito ("sin vencimiento") al tipar como `DATE` |
| Fechas con formato inválido / futuras | RUES | _ver salida_ | Cuarentena o corrección en Silver, no en Bronze |
| `valor <= 0` en TRM | TRM | _ver salida_ | Investigar y excluir en Silver si son errores de origen |

> Importante: ninguna de estas anomalías se corrige en este notebook. La Capa Bronze conserva el dato **tal cual llegó de la fuente**, cumpliendo el principio de inmutabilidad.